# Text-to-video gratuit — Google Colab (sans GPU chez vous)

**Modèle :** Wan 2.1 T2V 1.3B (~8 Go VRAM, compatible T4 Colab)  
**Coût :** 0 € (GPU Colab + Mistral Free Tier optionnel)

## Avant de commencer

1. **Exécution → Modifier le type d'exécution → GPU (T4)**
2. Puis **Exécution → Tout exécuter**

Les MP4 finaux sont dans `/content/output/`.

In [ ]:
import torch
assert torch.cuda.is_available(), "Activez le GPU : Exécution → Modifier le type d'exécution → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (Go):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
!pip -q install "torch>=2.1" diffusers transformers accelerate safetensors imageio imageio-ffmpeg huggingface_hub
!apt-get -qq install -y ffmpeg

In [ ]:
# ═══ CONFIGURATION — modifiez ici (ou collez depuis Video Factory) ═══
TOPIC = "Les secrets de la productivité"
SCRIPT = ""  # Collez votre script complet ici (prioritaire sur TOPIC)
DURATION_MIN = 1
CLIP_SEC = 4
ASPECT = "9:16"
BATCH_COUNT = 1
FPS = 16

MISTRAL_API_KEY = ""
HF_TOKEN = ""

RESOLUTIONS = {"9:16": (480, 832), "16:9": (832, 480), "1:1": (512, 512)}
WIDTH, HEIGHT = RESOLUTIONS.get(ASPECT, (480, 832))
NUM_FRAMES = max(17, ((FPS * CLIP_SEC) // 4) * 4 + 1)

In [ ]:
import json
import re
import subprocess
from pathlib import Path

import requests
from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline
from diffusers.utils import export_to_video

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
NEGATIVE = (
    "worst quality, inconsistent motion, blurry, static, text overlay, watermark, "
    "logo, ugly, deformed, low quality"
)

SCENE_TEMPLATES = [
    "Cinematic wide establishing shot of {topic}, dramatic lighting, film grain",
    "Dynamic medium shot exploring {topic}, smooth camera movement, rich colors",
    "Close-up detail about {topic}, shallow depth of field, moody ambiance",
    "Aerial perspective on {topic}, sweeping motion, golden hour lighting",
    "Slow tracking shot through environment related to {topic}, volumetric light",
]


def split_script_chunks(script, scene_count):
    text = (script or "").strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    chunks = paragraphs if len(paragraphs) >= 2 else []
    if len(chunks) < scene_count:
        sentences = [s.strip() for s in re.split(r"(?<=[.!?…])\s+", text) if s.strip()]
        if len(sentences) >= scene_count:
            chunks = []
            per = max(1, (len(sentences) + scene_count - 1) // scene_count)
            for i in range(scene_count):
                part = " ".join(sentences[i * per : (i + 1) * per]).strip()
                if part:
                    chunks.append(part)
    if not chunks:
        words = text.split()
        per = max(1, (len(words) + scene_count - 1) // scene_count)
        for i in range(scene_count):
            chunks.append(" ".join(words[i * per : (i + 1) * per]).strip())
    while len(chunks) > scene_count:
        chunks[-2] = f"{chunks[-2]} {chunks[-1]}".strip()
        chunks.pop()
    while len(chunks) < scene_count:
        chunks.append(chunks[-1] if chunks else text[:240])
    return chunks[:scene_count]


def plan_scenes_free(topic, duration_min, clip_sec):
    count = max(1, int((duration_min * 60) / clip_sec))
    scenes = []
    for i in range(count):
        tpl = SCENE_TEMPLATES[i % len(SCENE_TEMPLATES)]
        scenes.append({"index": i + 1, "visualPrompt": tpl.replace("{topic}", topic)})
    return {"title": topic[:80], "scenes": scenes}


def plan_scenes_mistral(topic, duration_min, clip_sec, api_key):
    count = max(1, int((duration_min * 60) / clip_sec))
    prompt = f"""Tu es directeur de création vidéo. Plan JSON pour {duration_min} min.
Sujet: {topic}
Format: {ASPECT}
Scènes EXACTES: {count}, {clip_sec}s chacune.
Prompts visuels en anglais, narration en français, pas de logos ni célébrités.
Réponds UNIQUEMENT en JSON:
{{"title":"...","scenes":[{{"index":1,"visualPrompt":"...","narration":"...","durationSec":{clip_sec}}}]}}"""
    res = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": "mistral-small-2506",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7,
        },
        timeout=120,
    )
    res.raise_for_status()
    raw = res.json()["choices"][0]["message"]["content"]
    clean = re.sub(r"```json|```", "", raw).strip()
    match = re.search(r"\{[\s\S]*\}", clean)
    if not match:
        raise ValueError("Réponse Mistral invalide")
    return json.loads(match.group(0))


def plan_scenes_from_script(script, duration_min, clip_sec):
    count = max(1, int((duration_min * 60) / clip_sec))
    chunks = split_script_chunks(script, count)
    title = script.split("\n")[0].strip()[:80] or "Vidéo personnalisée"
    scenes = []
    for i, narration in enumerate(chunks):
        tpl = SCENE_TEMPLATES[i % len(SCENE_TEMPLATES)]
        scenes.append({
            "index": i + 1,
            "visualPrompt": tpl.replace("{topic}", narration[:160]),
            "narration": narration,
        })
    return {"title": title, "scenes": scenes}


def plan_scenes_mistral_script(script, duration_min, clip_sec, api_key):
    count = max(1, int((duration_min * 60) / clip_sec))
    prompt = f"""Tu es directeur vidéo. Script utilisateur:
"""
{script[:12000]}
"""
Durée: {duration_min} min, format {ASPECT}, {clip_sec}s/scène, EXACTEMENT {count} scènes.
JSON uniquement: {{"title":"...","scenes":[{{"index":1,"visualPrompt":"...","narration":"...","durationSec":{clip_sec}}}]}}"""
    res = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": "mistral-small-2506", "messages": [{"role": "user", "content": prompt}], "temperature": 0.65},
        timeout=120,
    )
    res.raise_for_status()
    raw = res.json()["choices"][0]["message"]["content"]
    clean = re.sub(r"```json|```", "", raw).strip()
    match = re.search(r"\{[\s\S]*\}", clean)
    if not match:
        raise ValueError("Réponse Mistral invalide")
    return json.loads(match.group(0))


def plan_scenes(topic, duration_min, clip_sec):
    script = (globals().get("SCRIPT") or "").strip()
    if script:
        if MISTRAL_API_KEY.strip():
            try:
                return plan_scenes_mistral_script(script, duration_min, clip_sec, MISTRAL_API_KEY.strip())
            except Exception as exc:
                print("Mistral script indisponible, fallback découpage:", exc)
        return plan_scenes_from_script(script, duration_min, clip_sec)
    if MISTRAL_API_KEY.strip():
        try:
            return plan_scenes_mistral(topic, duration_min, clip_sec, MISTRAL_API_KEY.strip())
        except Exception as exc:
            print("Mistral indisponible, fallback templates:", exc)
    return plan_scenes_free(topic, duration_min, clip_sec)


print("Chargement du modèle Wan 2.1 (première fois: ~10-20 min)...")
token = HF_TOKEN.strip() or None
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32, token=token)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16, token=token)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("Modèle prêt.")

In [ ]:
def generate_clip(prompt, out_path, seed=42):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=40,
        guidance_scale=6.0,
        generator=gen,
    )
    export_to_video(result.frames[0], str(out_path), fps=FPS)
    return out_path


def concat_clips(clip_paths, out_path):
    list_file = out_path.with_suffix(".txt")
    list_file.write_text("\n".join(f"file '{p}'" for p in clip_paths), encoding="utf-8")
    subprocess.run(
        ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_file), "-c", "copy", str(out_path)],
        check=True,
        capture_output=True,
    )
    list_file.unlink(missing_ok=True)
    return out_path


def produce_video(topic, batch_index=0):
    script = (globals().get("SCRIPT") or "").strip()
    plan = plan_scenes(topic, DURATION_MIN, CLIP_SEC)
    label = plan.get("title") or topic
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", label)[:40]
    work = OUTPUT_DIR / f"batch{batch_index}_{safe}"
    work.mkdir(parents=True, exist_ok=True)
    clips = []
    for scene in plan["scenes"]:
        idx = scene.get("index", len(clips) + 1)
        prompt = scene.get("visualPrompt") or scene.get("prompt") or topic
        clip_path = work / f"scene_{idx:02d}.mp4"
        print(f"  Scène {idx}/{len(plan['scenes'])}: {prompt[:60]}...")
        generate_clip(prompt, clip_path, seed=42 + idx)
        clips.append(clip_path)
    final = work / "final.mp4"
    if len(clips) == 1:
        final.write_bytes(clips[0].read_bytes())
    else:
        concat_clips(clips, final)
    print(f"✓ Vidéo prête: {final}")
    return final


topics = [TOPIC] if BATCH_COUNT <= 1 else [f"{TOPIC} — partie {i+1}" for i in range(BATCH_COUNT)]
if (globals().get("SCRIPT") or "").strip():
    topics = [plan_scenes(TOPIC, DURATION_MIN, CLIP_SEC).get("title", TOPIC)]
results = []
for i, t in enumerate(topics):
    print(f"\n=== Vidéo {i+1}/{len(topics)}: {t} ===")
    results.append(produce_video(t, batch_index=i))

print("\n=== TERMINÉ ===")
for r in results:
    print(r)

In [ ]:
from google.colab import files

for mp4 in sorted(OUTPUT_DIR.rglob("final.mp4")):
    print("Téléchargement:", mp4.name)
    files.download(str(mp4))